In [1]:
import os
import tensorflow as tf
from tensorflow.keras import mixed_precision, layers, models, optimizers, losses, metrics
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import glob
import time
import csv
from datetime import datetime
import matplotlib.pyplot as plt

# Reduce TF log messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Enable GPU memory growth
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

# Use mixed precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision enabled:", mixed_precision.global_policy())

2026-01-29 16:48:53.879905: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-29 16:48:53.897450: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-29 16:48:53.918666: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-29 16:48:53.925400: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-29 16:48:53.942086: I tensorflow/core/platform/cpu_feature_guar

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, compute capability 12.0
Mixed precision enabled: <Policy "mixed_float16">


I0000 00:00:1769705336.914180    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705336.981979    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705336.987179    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705336.989247    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [2]:
# # Parse .tfrecords files
# # Image stored as raw bytes and label stored as int (0, 1, 2)
# feature_description = {
#     "image": tf.io.FixedLenFeature([], tf.string),
#     "label": tf.io.FixedLenFeature([], tf.int64),
# }
# }

# def parse_example(example_proto):
#     example = tf.io.parse_single_example(example_proto, feature_description)
#     image = tf.io.decode_raw(example["image"], tf.uint8)
#     image = tf.reshape(image, [299, 299, 1])
#     image = tf.cast(image, tf.float32) / 255.0  # normalize
#     label = tf.cast(example["label"], tf.int32)
#     return image, label

# # Collect all .tfrecords in the training folder
# # train_folder = "Kaggle_Data/Correct_Training_Data/"
# train_folder = "Kaggle_Data_recheck/Cleaned_Training_Data/" # second data cleaning
# tfrecord_files = glob.glob(os.path.join(train_folder, "*.tfrecords"))

# # Load dataset from all files
# raw_train_ds = tf.data.TFRecordDataset(tfrecord_files)
# raw_train_ds = raw_train_ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)

# # Inspect samples
# for img, lbl in raw_train_ds.take(3):
#     print("Image shape:", img.shape, "Label:", lbl.numpy())

In [3]:
# # Training pipeline
# BATCH_SIZE = 32
# SHUFFLE_BUFFER = 1000

# train_ds = (
#     raw_train_ds
#     .shuffle(SHUFFLE_BUFFER)
#     .batch(BATCH_SIZE)
#     .prefetch(tf.data.AUTOTUNE)
# )

# # Verify dimensions
# for images, labels in train_ds.take(1):
#     print("Batch image shape:", images.shape)
#     print("Batch label shape:", labels.shape)

In [4]:
# # Create a balanced dataset that is a subset of the original

# SEED = 0

# # Load entire dataset into memory
# images = []
# labels = []

# for img, lbl in raw_train_ds:
#     images.append(img.numpy())
#     labels.append(lbl.numpy())

# images = np.stack(images)
# labels = np.array(labels)

# print("Total samples:", len(labels))

# # Split by class
# idx_0 = np.where(labels == 0)[0]
# idx_1 = np.where(labels == 1)[0]
# idx_2 = np.where(labels == 2)[0]

# print(f"Class counts before balancing:")
# print(f"  Class 0: {len(idx_0)}")
# print(f"  Class 1: {len(idx_1)}")
# print(f"  Class 2: {len(idx_2)}")

# # Use smallest class as reference
# n = len(idx_2)

# rng = np.random.default_rng(SEED)

# idx_0_bal = rng.choice(idx_0, size=n, replace=False)
# idx_1_bal = rng.choice(idx_1, size=n, replace=False)

# balanced_idx = np.concatenate([idx_0_bal, idx_1_bal, idx_2])

# # Shuffle
# rng.shuffle(balanced_idx)

# images_bal = images[balanced_idx]
# labels_bal = labels[balanced_idx]

# print(f"\nBalanced dataset size: {len(labels_bal)}")
# print("Samples per class:", np.bincount(labels_bal))

# np.save("images_bal.npy", images_bal)
# np.save("labels_bal.npy", labels_bal)

# Load balanced dataset
images_bal = np.load("images_bal.npy")
labels_bal = np.load("labels_bal.npy")

BATCH_SIZE = 32
SHUFFLE_BUFFER = 1000

# Rebuild tf.data pipeline
train_ds = (
    tf.data.Dataset.from_tensor_slices((images_bal, labels_bal))
    .shuffle(buffer_size=SHUFFLE_BUFFER, seed=0)  # randomizes order within the buffer
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Verification
for imgs, lbls in train_ds.take(1):
    print("\nBatch image shape:", imgs.shape)
    print("Batch label counts:", np.bincount(lbls.numpy()))

I0000 00:00:1769705338.408785    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705338.411363    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705338.413580    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769705338.547196    1931 cuda_executor.cc:1015] successful NUMA node read from SysFS ha


Batch image shape: (32, 299, 299, 1)
Batch label counts: [14  8 10]


2026-01-29 16:49:05.453345: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
# Model architecture
model = models.Sequential([
    layers.Input(shape=(299, 299, 1)),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    # layers.Conv2D(512, 3, padding="same", activation="relu"),
    # layers.Conv2D(512, 3, padding="same", activation="relu"),
    # layers.MaxPooling2D(),
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(3, activation="softmax")

   
])
# Original Model
# model = models.Sequential([
#     layers.Input(shape=(299, 299, 1)),

#     layers.Conv2D(32, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(64, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(128, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.Conv2D(256, 3, padding="same", activation="relu"),
#     layers.BatchNormalization(),
#     layers.MaxPooling2D(),

#     layers.GlobalAveragePooling2D(),
#     layers.Dropout(0.4),

#     layers.Dense(3, activation="softmax")
# ])

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss=losses.SparseCategoricalCrossentropy(),
    metrics=[
        metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 299, 299, 32)      320       
                                                                 
 conv2d_1 (Conv2D)           (None, 299, 299, 32)      9248      
                                                                 
 max_pooling2d (MaxPooling2  (None, 149, 149, 32)      0         
 D)                                                              
                                                                 
 conv2d_2 (Conv2D)           (None, 149, 149, 64)      18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 149, 149, 64)      36928     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 74, 74, 64)        0         
 g2D)                                                   

In [6]:
# Recompute training labels as NumPy array
# train_labels = []

# for _, lbl in raw_train_ds:
#     train_labels.append(lbl.numpy())

# train_labels = np.array(train_labels)

# print("Training label distribution:")
# unique, counts = np.unique(train_labels, return_counts=True)
# for u, c in zip(unique, counts):
#     print(f"Class {u}: {c}")

# # Compute class weights
# class_weights = compute_class_weight( 
#     class_weight="balanced",
#     classes=np.array([0, 1, 2]),
#     y=train_labels
# )

# class_weights = dict(enumerate(class_weights))
# print("\nClass weights:", class_weights)

# Load validation data from .npy files
# x_val = np.load("Kaggle_Data/Correct_Validation_Data/validation_data.npy")
# y_val = np.load("Kaggle_Data/Correct_Validation_Data/validation_labels.npy")

x_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_data.npy")
y_val = np.load("Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_labels.npy")

# Verify channel dimension
if x_val.ndim == 3:
    x_val = x_val[..., np.newaxis]

x_val = x_val.astype("float32") / 255.0
y_val = y_val.astype("int32")

print("\nValidation images shape:", x_val.shape)
print("Validation labels shape:", y_val.shape)




Validation images shape: (7682, 299, 299, 1)
Validation labels shape: (7682,)


In [7]:
# Train model
EPOCHS = 20

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

# class_weights = {
#     0: 1.0,   # Normal
#     1: 2.5,   # Benign
#     2: 3    # Malignant
# }

start_time = time.time()
history = model.fit( 
    train_ds,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    # class_weight=class_weights,
    callbacks=[early_stop]
)
end_time = time.time()
training_time_sec = end_time - start_time


Epoch 1/20


2026-01-29 16:49:11.365536: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
2026-01-29 16:49:11.473199: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90700
W0000 00:00:1769705351.558304    2041 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705351.559781    2041 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705351.569895    2041 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705351.571830    2041 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced

  1/308 [..............................] - ETA: 41:53 - loss: 1.0996 - accuracy: 0.2812

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1769705356.317387    2021 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769

  2/308 [..............................] - ETA: 13:06 - loss: 1.0996 - accuracy: 0.2969

...

308/308 [==============================] - ETA: 0s - loss: 0.9296 - accuracy: 0.5208 

W0000 00:00:1769705422.408201    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408398    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408491    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408580    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408662    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408738    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408812    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408900    2028 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705422.408981    2028 gp

308/308 [==============================] - 21s 69ms/step - loss: 0.9296 - accuracy: 0.5208 - val_loss: 0.7855 - val_accuracy: 0.6226
Epoch 4/20
308/308 [==============================] - ETA: 0s - loss: 0.9134 - accuracy: 0.5309 

W0000 00:00:1769705443.696292    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696447    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696544    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696636    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696727    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696817    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.696914    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.697016    2027 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1769705443.697142    2027 gp

308/308 [==============================] - 21s 69ms/step - loss: 0.9134 - accuracy: 0.5309 - val_loss: 0.7013 - val_accuracy: 0.6850
Epoch 5/20
308/308 [==============================] - 21s 69ms/step - loss: 0.9092 - accuracy: 0.5391 - val_loss: 0.6490 - val_accuracy: 0.7236
Epoch 6/20
308/308 [==============================] - 21s 69ms/step - loss: 0.9066 - accuracy: 0.5386 - val_loss: 0.6633 - val_accuracy: 0.7143
Epoch 7/20
308/308 [==============================] - 21s 69ms/step - loss: 0.9045 - accuracy: 0.5479 - val_loss: 0.7336 - val_accuracy: 0.6934
Epoch 8/20
308/308 [==============================] - 21s 69ms/step - loss: 0.8929 - accuracy: 0.5511 - val_loss: 0.6850 - val_accuracy: 0.7105
Epoch 9/20
308/308 [==============================] - 21s 69ms/step - loss: 0.8765 - accuracy: 0.5635 - val_loss: 0.8807 - val_accuracy: 0.6269
Epoch 10/20
308/308 [==============================] - 21s 68ms/step - loss: 0.8909 - accuracy: 0.5565 - val_loss: 0.6027 - val_accuracy: 0.7722
Ep

In [8]:
# Make predictions on validation set
val_ds = (
    tf.data.Dataset.from_tensor_slices(x_val)
    .batch(32)
)
y_pred_probs = model.predict(val_ds)


# y_pred_probs = model.predict(x_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred)
print("Confusion matrix:\n", cm)

# Generate normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Report per-class metrics
report = classification_report(y_val, y_pred, target_names=["Normal", "Benign", "Malignant"])

report_dict = classification_report(
    y_val,
    y_pred,
    target_names=["Normal", "Benign", "Malignant"],
    output_dict=True
)

print("\nClassification report:\n", report)

# Report per-class accuracy manually
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({['Normal','Benign','Malignant'][i]}): {acc:.2f}")

241/241 [==============================] - 7s 18ms/step
Confusion matrix:
 [[5338  986  339]
 [ 117  337  139]
 [  61  209  156]]

Normalized confusion matrix:
 [[0.8  0.15 0.05]
 [0.2  0.57 0.23]
 [0.14 0.49 0.37]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.97      0.80      0.88      6663
      Benign       0.22      0.57      0.32       593
   Malignant       0.25      0.37      0.29       426

    accuracy                           0.76      7682
   macro avg       0.48      0.58      0.50      7682
weighted avg       0.87      0.76      0.80      7682

Accuracy for class 0 (Normal): 0.80
Accuracy for class 1 (Benign): 0.57
Accuracy for class 2 (Malignant): 0.37


In [9]:
CSV_FILE = "experiments_v2.csv"
CLASS_NAMES = ["Normal", "Benign", "Malignant"]

# Auto-increment experiment ID
def get_next_experiment_id(csv_file):
    if not os.path.exists(csv_file):
        return 1
    with open(csv_file, "r") as f:
        return sum(1 for _ in f)  # header + rows

# Extract max L1/L2 across layers
def extract_l1_l2(model):
    l1_vals = []
    l2_vals = []
    for layer in model.layers:
        reg = getattr(layer, "kernel_regularizer", None)
        if reg is not None:
            if hasattr(reg, "l1") and reg.l1 is not None:
                l1_vals.append(float(tf.keras.backend.get_value(reg.l1)))
            if hasattr(reg, "l2") and reg.l2 is not None:
                l2_vals.append(float(tf.keras.backend.get_value(reg.l2)))
    return (
        max(l1_vals) if l1_vals else 0.0,
        max(l2_vals) if l2_vals else 0.0
    )

def extract_max_dropout(model):
    rates = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dropout):
            rates.append(float(layer.rate))
    return max(rates) if rates else 0.0

# Build experiment row
exp_id = get_next_experiment_id(CSV_FILE)

per_class_acc = cm.diagonal() / cm.sum(axis=1)

l1_val, l2_val = extract_l1_l2(model)
dropout_val = extract_max_dropout(model)
learning_rate_val = float(model.optimizer.learning_rate.numpy())

# Final losses and accuracies from training history
train_loss = history.history["loss"][-1]
val_loss = history.history["val_loss"][-1]
train_acc = history.history["accuracy"][-1]
val_acc = history.history["val_accuracy"][-1]

row = {
    # Metadata
    "experiment_id": exp_id,
    "timestamp": datetime.now().isoformat(),

    # Training parameters
    "batch_size": BATCH_SIZE,
    "shuffle_buffer": SHUFFLE_BUFFER,
    "epochs": EPOCHS,
    "learning_rate": learning_rate_val,
    "dropout_rate": dropout_val,
    "l1": l1_val,
    "l2": l2_val,

    # Performance metrics
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,

    "training_time_min": training_time_sec / 60.0
}

for i, cls in enumerate(CLASS_NAMES):
    cls_l = cls.lower()
    row[f"acc_{cls_l}"] = per_class_acc[i]
    row[f"precision_{cls_l}"] = report_dict[cls]["precision"]
    row[f"recall_{cls_l}"] = report_dict[cls]["recall"]
    row[f"f1_{cls_l}"] = report_dict[cls]["f1-score"]

file_exists = os.path.isfile(CSV_FILE)

with open(CSV_FILE, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Experiment {exp_id} logged to {CSV_FILE}")

# Ensure folder exists
LOSS_PLOT_FOLDER = "loss_plots"
os.makedirs(LOSS_PLOT_FOLDER, exist_ok=True)

# Plot training vs validation loss
plt.figure(figsize=(8, 6))
plt.plot(history.history["loss"], label="Training Loss", marker='o')
plt.plot(history.history["val_loss"], label="Validation Loss", marker='o')
plt.title(f"Experiment {exp_id:05d} Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Save the figure with experiment ID
plot_filename = os.path.join(LOSS_PLOT_FOLDER, f"{exp_id:05d}.png")
plt.savefig(plot_filename)
plt.close()  # Close figure to free memory

print(f"Loss plot saved to {plot_filename}")


Experiment 129 logged to experiments_v2.csv
Loss plot saved to loss_plots/00129.png
